# Ablation Study: Tháo từng thành phần để đánh giá ảnh hưởng

Mỗi ablation tháo **1 thành phần** khỏi full model, giữ nguyên tất cả còn lại.
Full model (checkpoint đã có) làm baseline để so sánh.

| Ablation | Thành phần tháo | Thay thế bằng |
|----------|----------------|---------------|
| 1 | Adaptive Filtering (`filter_threshold`) | Uniform pooling |
| 3 | Narration Auxiliary Loss (`lambda_narr`) | `lambda_narr = 0.0` |
| 4 | Gated Attention (sigmoid gate) | Standard attention (gate = 1) |
| MRL | MRL Loss (`gamma_mrl`) | `gamma_mrl = 0.0` |

Checkpoints lưu tại `source/ablation_output/`.

In [1]:
import gc
import json
import math
import os
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_cosine_schedule_with_warmup

cwd = Path.cwd().resolve()
if (cwd / 'source').exists():
    BASE_DIR = cwd
elif (cwd.parent / 'source').exists():
    BASE_DIR = cwd.parent
else:
    raise RuntimeError(f'Cannot locate project root from cwd={cwd}')

SOURCE_DIR = BASE_DIR / 'source'
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('BASE_DIR:', BASE_DIR)
print('DEVICE  :', DEVICE)
if torch.cuda.is_available():
    print('GPU     :', torch.cuda.get_device_name(0))

/home/urlab/miniconda3/envs/uav_ai/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


BASE_DIR: /media/urlab/KINGSTON/aic
DEVICE  : cuda
GPU     : NVIDIA GeForce RTX 5060 Ti


In [2]:
from dataset import get_dataloader
from model import FusionEncoder, FusionBlock, GatedAttention

AIC_ROOT_HARDCODE = Path('/media/urlab/KINGSTON/aic')

ABLATION_OUTPUT = SOURCE_DIR / 'ablation_output'
ABLATION_OUTPUT.mkdir(parents=True, exist_ok=True)

FULL_MODEL_CKPT = AIC_ROOT_HARDCODE / 'source' / 'fusion_output' / 'best_fusion_model.pth'
BGEM3_PATH = AIC_ROOT_HARDCODE / 'features' / 'weights' / 'bgem3'
HF_CACHE_DIR = AIC_ROOT_HARDCODE / 'features' / 'weights' / 'hf_cache'

CLIP_MODEL_ID = 'openai/clip-vit-base-patch16'
CLIP_LAYER_IDX = -2
CLIP_VIS_DIM = 768
CLIP_FEATURE_SUBDIR = f'clip_vitb16_l{abs(CLIP_LAYER_IDX)}'
CLIP_FEATURE_DIR = AIC_ROOT_HARDCODE / 'features' / CLIP_FEATURE_SUBDIR

CFG = {
    'base_dir': AIC_ROOT_HARDCODE,
    # data
    'max_frames': 25,
    'max_segments': 15,
    'max_seg_tokens': 128,
    # model
    'dim': 1024,
    'vis_dim': 1152,
    'n_layers': 2,
    'n_heads': 16,
    'n_kv_heads': 4,
    # train
    'epochs': 20,
    'batch_size': 4,
    'num_workers': os.cpu_count() or 4,
    'lr': 1e-4,
    'weight_decay': 0.01,
    'warmup_ratio': 0.1,
    'grad_clip': 1.0,
    'train_query_max_length': 512,
    # eval
    'dual_softmax_tau': 0.01,
    'eval_query_max_length': 256,
    'eval_query_batch_size': 16,
    # visual backend config
    'visual_feature_subdir': 'siglip2',
    'clip_model_id': CLIP_MODEL_ID,
    'clip_layer_idx': CLIP_LAYER_IDX,
    'clip_vis_dim': CLIP_VIS_DIM,
    'clip_feature_subdir': CLIP_FEATURE_SUBDIR,
    'clip_train_from_scratch': True,
}

print('AIC root (hardcoded):', AIC_ROOT_HARDCODE)
print('Ablation output dir  :', ABLATION_OUTPUT)
print('Full model ckpt      :', FULL_MODEL_CKPT)
print('HF cache dir         :', HF_CACHE_DIR)
print('CLIP model id        :', CLIP_MODEL_ID)
print('CLIP feature dir     :', CLIP_FEATURE_DIR)

AIC root (hardcoded): /media/urlab/KINGSTON/aic
Ablation output dir  : /media/urlab/KINGSTON/aic/source/ablation_output
Full model ckpt      : /media/urlab/KINGSTON/aic/source/fusion_output/best_fusion_model.pth
HF cache dir         : /media/urlab/KINGSTON/aic/features/weights/hf_cache
CLIP model id        : openai/clip-vit-base-patch16
CLIP feature dir     : /media/urlab/KINGSTON/aic/features/clip_vitb16_l2


In [3]:
train_loader = get_dataloader(
    split='train', base_dir=CFG['base_dir'],
    batch_size=CFG['batch_size'], num_workers=CFG['num_workers'],
    max_frames=CFG['max_frames'], max_segments=CFG['max_segments'],
    max_seg_tokens=CFG['max_seg_tokens'],
    visual_feature_subdir=CFG['visual_feature_subdir'],
)
val_loader = get_dataloader(
    split='val', base_dir=CFG['base_dir'],
    batch_size=CFG['batch_size'], num_workers=CFG['num_workers'],
    max_frames=CFG['max_frames'], max_segments=CFG['max_segments'],
    max_seg_tokens=CFG['max_seg_tokens'],
    visual_feature_subdir=CFG['visual_feature_subdir'],
)

clip_train_loader = None
clip_val_loader = None
if CLIP_FEATURE_DIR.exists():
    clip_train_loader = get_dataloader(
        split='train', base_dir=CFG['base_dir'],
        batch_size=CFG['batch_size'], num_workers=CFG['num_workers'],
        max_frames=CFG['max_frames'], max_segments=CFG['max_segments'],
        max_seg_tokens=CFG['max_seg_tokens'],
        visual_feature_subdir=CFG['clip_feature_subdir'],
    )
    clip_val_loader = get_dataloader(
        split='val', base_dir=CFG['base_dir'],
        batch_size=CFG['batch_size'], num_workers=CFG['num_workers'],
        max_frames=CFG['max_frames'], max_segments=CFG['max_segments'],
        max_seg_tokens=CFG['max_seg_tokens'],
        visual_feature_subdir=CFG['clip_feature_subdir'],
    )
else:
    print(f'[WARN] CLIP feature dir not found: {CLIP_FEATURE_DIR}')
    print('       Run source/precompute_clip_patch16.py first to build CLIP features.')

bgem3_tokenizer = AutoTokenizer.from_pretrained(str(BGEM3_PATH), local_files_only=True)
bgem3_model = AutoModel.from_pretrained(str(BGEM3_PATH), local_files_only=True)
bgem3_model.eval().to(DEVICE)
for p in bgem3_model.parameters():
    p.requires_grad = False

print('train samples:', len(train_loader.dataset))
print('val   samples:', len(val_loader.dataset))
if clip_train_loader is not None:
    print('clip train samples:', len(clip_train_loader.dataset))
    print('clip val   samples:', len(clip_val_loader.dataset))
else:
    print('clip loaders      : not ready')
print('BGE-M3 loaded from:', BGEM3_PATH)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 516.32it/s, Materializing param=pooler.dense.weight]                               


train samples: 9222
val   samples: 9222
clip train samples: 9222
clip val   samples: 9222
BGE-M3 loaded from: /media/urlab/KINGSTON/aic/features/weights/bgem3


## Định nghĩa Ablation Model Variants

In [4]:
# -- Ablation 4: No Gated Attention -------------------------------------------
class GatedAttentionNoGate(GatedAttention):
    """Bỏ sigmoid gate -> output = P@V (không nhân thêm hệ số gate)."""
    def forward(self, x_q, x_kv, pos_q, pos_k, mask=None):
        q = self._shape_q(self.w_q(x_q))
        k = self._shape_kv(self.w_k(x_kv))
        v = self._shape_kv(self.w_v(x_kv))
        k = k.repeat_interleave(self.kv_repeat, dim=1)
        v = v.repeat_interleave(self.kv_repeat, dim=1)
        q = self.rope.apply(q, pos_q)
        k = self.rope.apply(k, pos_k)
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if mask is not None:
            if mask.dim() == 2:
                scores = scores.masked_fill(mask.unsqueeze(1).unsqueeze(1) == 0, float('-inf'))
            elif mask.dim() == 3:
                scores = scores.masked_fill(mask.unsqueeze(1) == 0, float('-inf'))
        p = torch.softmax(scores, dim=-1)
        o = torch.matmul(p, v)  # khong co gate
        o = o.transpose(1, 2).contiguous().view(x_q.shape[0], x_q.shape[1], self.dim)
        return self.w_o(o)


class FusionBlockNoGate(FusionBlock):
    def __init__(self, dim=1024, n_heads=16, n_kv_heads=4, ffn_mult=8.0 / 3.0):
        super().__init__(dim, n_heads, n_kv_heads, ffn_mult)
        self.self_attn = GatedAttentionNoGate(dim, n_heads, n_kv_heads)
        self.cross_attn = GatedAttentionNoGate(dim, n_heads, n_kv_heads)


class FusionEncoderNoGate(FusionEncoder):
    """Ablation 4: Thay GatedAttention bằng standard attention."""
    def __init__(self, dim=1024, vis_dim=1152, n_layers=2, n_heads=16, n_kv_heads=4, delta=100.0):
        super().__init__(dim, vis_dim, n_layers, n_heads, n_kv_heads, delta)
        self.blocks = nn.ModuleList([
            FusionBlockNoGate(dim=dim, n_heads=n_heads, n_kv_heads=n_kv_heads)
            for _ in range(n_layers)
        ])


# -- Ablation 1: No Adaptive Filtering -----------------------------------------
class FusionEncoderNoFilter(FusionEncoder):
    """Ablation 1: Bỏ adaptive filtering, dùng uniform pooling."""
    def forward(
        self,
        seg_tokens, seg_pooled, visual_features,
        seg_timestamps, frame_timestamps,
        seg_mask, visual_mask,
        query_emb=None, token_mask=None,
    ):
        b, m, t, d = seg_tokens.shape
        _, n, r, _ = visual_features.shape

        e_narr = self._masked_mean(seg_pooled, seg_mask, dim=1)
        e_narr = F.normalize(e_narr, p=2, dim=-1)

        vis = self.visual_proj(visual_features)
        seg_pos, frame_pos = self._build_positions(
            seg_timestamps, frame_timestamps, t_tokens=t, n_regions=r
        )

        seg_flat = seg_tokens.reshape(b, m * t, d)
        vis_flat = vis.reshape(b, n * r, d)
        seg_pos_flat = seg_pos.reshape(b, m * t)
        frame_pos_flat = frame_pos.reshape(b, n * r)

        if token_mask is None:
            seg_token_mask = seg_mask.unsqueeze(-1).expand(b, m, t).reshape(b, m * t)
        else:
            seg_token_mask = token_mask.reshape(b, m * t)
        visual_region_mask = visual_mask.unsqueeze(-1).expand(b, n, r).reshape(b, n * r)

        for block in self.blocks:
            seg_flat = block(
                seg_tokens=seg_flat, visual_regions=vis_flat,
                seg_pos=seg_pos_flat, frame_pos=frame_pos_flat,
                seg_mask=seg_token_mask, visual_mask=visual_region_mask,
            )

        seg_flat = self.final_norm(seg_flat)
        seg_tokens_out = seg_flat.reshape(b, m, t, d)
        seg_token_mask_4d = seg_token_mask.reshape(b, m, t)
        seg_repr = self._masked_mean(seg_tokens_out, seg_token_mask_4d, dim=2)

        # Uniform pooling (không dùng query để filter)
        valid_seg = seg_mask > 0
        weights = valid_seg.float()
        weights = weights / weights.sum(dim=1, keepdim=True).clamp_min(1e-6)

        e_plus = torch.sum(seg_repr * weights.unsqueeze(-1), dim=1)
        e_plus = F.normalize(e_plus, p=2, dim=-1)
        return e_plus, e_narr, self.temperature


class FusionEncoderNoFilterNoGate(FusionEncoderNoGate):
    """Combo ablation: bỏ adaptive filtering + bỏ gated attention."""
    def forward(
        self,
        seg_tokens, seg_pooled, visual_features,
        seg_timestamps, frame_timestamps,
        seg_mask, visual_mask,
        query_emb=None, token_mask=None,
    ):
        b, m, t, d = seg_tokens.shape
        _, n, r, _ = visual_features.shape

        e_narr = self._masked_mean(seg_pooled, seg_mask, dim=1)
        e_narr = F.normalize(e_narr, p=2, dim=-1)

        vis = self.visual_proj(visual_features)
        seg_pos, frame_pos = self._build_positions(
            seg_timestamps, frame_timestamps, t_tokens=t, n_regions=r
        )

        seg_flat = seg_tokens.reshape(b, m * t, d)
        vis_flat = vis.reshape(b, n * r, d)
        seg_pos_flat = seg_pos.reshape(b, m * t)
        frame_pos_flat = frame_pos.reshape(b, n * r)

        if token_mask is None:
            seg_token_mask = seg_mask.unsqueeze(-1).expand(b, m, t).reshape(b, m * t)
        else:
            seg_token_mask = token_mask.reshape(b, m * t)
        visual_region_mask = visual_mask.unsqueeze(-1).expand(b, n, r).reshape(b, n * r)

        for block in self.blocks:
            seg_flat = block(
                seg_tokens=seg_flat, visual_regions=vis_flat,
                seg_pos=seg_pos_flat, frame_pos=frame_pos_flat,
                seg_mask=seg_token_mask, visual_mask=visual_region_mask,
            )

        seg_flat = self.final_norm(seg_flat)
        seg_tokens_out = seg_flat.reshape(b, m, t, d)
        seg_token_mask_4d = seg_token_mask.reshape(b, m, t)
        seg_repr = self._masked_mean(seg_tokens_out, seg_token_mask_4d, dim=2)

        valid_seg = seg_mask > 0
        weights = valid_seg.float()
        weights = weights / weights.sum(dim=1, keepdim=True).clamp_min(1e-6)

        e_plus = torch.sum(seg_repr * weights.unsqueeze(-1), dim=1)
        e_plus = F.normalize(e_plus, p=2, dim=-1)
        return e_plus, e_narr, self.temperature


print('Ablation model classes defined.')
print('  FusionEncoderNoFilter        - Ablation 1')
print('  FusionEncoderNoGate          - Ablation 4')
print('  FusionEncoderNoFilterNoGate  - Combo Ablation')
print('  FusionEncoder + lambda_narr=0  - Ablation 3')
print('  FusionEncoder + gamma_mrl=0    - MRL Test')

Ablation model classes defined.
  FusionEncoderNoFilter        - Ablation 1
  FusionEncoderNoGate          - Ablation 4
  FusionEncoderNoFilterNoGate  - Combo Ablation
  FusionEncoder + lambda_narr=0  - Ablation 3
  FusionEncoder + gamma_mrl=0    - MRL Test


## Shared Training & Evaluation Utilities

In [5]:

# Dimensions to test for MRL evaluation (smallest, middle, full)
MRL_DIMS = (64, 256, 1024)


def encode_queries(texts, tokenizer, encoder_model, device, max_length=512, batch_size=None):
    if not texts:
        return torch.empty((0, encoder_model.config.hidden_size), device=device)
    if batch_size is None:
        batch_size = len(texts)
    all_embs = []
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            chunk = texts[start:start + batch_size]
            enc = tokenizer(
                chunk, padding=True, truncation=True,
                max_length=max_length, return_tensors='pt',
            ).to(device)
            if device.type == 'cuda':
                with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                    h = encoder_model(**enc, return_dict=True).last_hidden_state
            else:
                h = encoder_model(**enc, return_dict=True).last_hidden_state
            m = enc['attention_mask'].unsqueeze(-1).to(h.dtype)
            pooled = (h * m).sum(1) / m.sum(1).clamp_min(1e-6)
            all_embs.append(F.normalize(pooled.float(), p=2, dim=-1))
    return torch.cat(all_embs, dim=0)


def symmetric_infonce(q, d, temperature_param):
    tau = torch.exp(temperature_param).clamp_min(1e-6)
    sim = torch.matmul(q, d.T) / tau
    labels = torch.arange(sim.size(0), device=sim.device)
    return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.T, labels))


def retrieval_loss(q_hat, e_plus, e_narr, temperature, hard_neg_embs=None,
                   lambda_narr=0.5, beta_hard=0.3, gamma_mrl=0.1):
    l_base = symmetric_infonce(q_hat, e_plus, temperature)

    l_narr = torch.tensor(0.0, device=q_hat.device)
    if lambda_narr > 0:
        l_narr = symmetric_infonce(q_hat, e_narr, temperature)

    l_hard = torch.tensor(0.0, device=q_hat.device)
    if hard_neg_embs is not None and beta_hard > 0:
        q = q_hat.unsqueeze(1)
        neg_scores = torch.sum(q * hard_neg_embs, dim=-1)
        pos_scores = torch.sum(q_hat * e_plus, dim=-1, keepdim=True)
        l_hard = torch.relu(neg_scores - pos_scores + 0.1).mean()

    l_mrl = torch.tensor(0.0, device=q_hat.device)
    if gamma_mrl > 0:
        for d in [64, 128, 256, 512, 1024]:
            l_mrl = l_mrl + symmetric_infonce(q_hat[:, :d], e_plus[:, :d], temperature)
        l_mrl = l_mrl / 5.0

    return l_base + lambda_narr * l_narr + beta_hard * l_hard + gamma_mrl * l_mrl


def _encode_hard_negs(hard_neg_texts_batch, tokenizer, encoder, device, max_length):
    b = len(hard_neg_texts_batch)
    k = max((len(x) for x in hard_neg_texts_batch), default=0)
    if k == 0:
        return None
    flat = []
    for negs in hard_neg_texts_batch:
        flat.extend(negs + [''] * (k - len(negs)))
    emb = encode_queries(flat, tokenizer, encoder, device, max_length=max_length)
    return emb.view(b, k, -1)


def precompute_val_docs(model, loader, device):
    model.eval()
    doc_embs, shot_ids = [], []
    with torch.inference_mode():
        for batch in tqdm(loader, desc='Val docs', leave=False, dynamic_ncols=True):
            with torch.amp.autocast(
                device_type='cuda' if device.type == 'cuda' else 'cpu',
                enabled=(device.type == 'cuda'), dtype=torch.bfloat16,
            ):
                e_plus, _, _ = model(
                    seg_tokens=batch['token_reprs'].to(device),
                    seg_pooled=batch['segment_pooled'].to(device),
                    visual_features=batch['visual_features'].to(device),
                    seg_timestamps=batch['seg_timestamps'].to(device),
                    frame_timestamps=batch['frame_timestamps'].to(device),
                    seg_mask=batch['segment_mask'].to(device),
                    visual_mask=batch['visual_mask'].to(device),
                    query_emb=None,
                    token_mask=batch['token_mask'].to(device),
                )
            doc_embs.append(e_plus.detach().cpu())
            shot_ids.extend(batch['shot_id'])
    return F.normalize(torch.cat(doc_embs, dim=0), p=2, dim=-1), shot_ids


def load_val_queries():
    q_items = []
    for jf in sorted((BASE_DIR / 'data' / 'val').glob('*.json')):
        video_id = jf.stem
        for row in json.loads(jf.read_text('utf-8')):
            sid = str(row.get('id', '')).zfill(3)
            q = str(row.get('positive', '')).strip()
            if sid and q:
                q_items.append({'shot_id': f'{video_id}_{sid}', 'query': q})
    seen, deduped = set(), []
    for x in q_items:
        if x['shot_id'] not in seen:
            seen.add(x['shot_id'])
            deduped.append(x)
    return deduped


def evaluate_on_val(model, device):
    """
    Evaluate on val split. Returns metrics at full dim + MRL_DIMS truncated dims.
    Queries are encoded once and reused across all dims.
    """
    doc_embs, shot_ids = precompute_val_docs(model, val_loader, device)
    q_items = load_val_queries()

    # Encode queries once
    q_embs_all = encode_queries(
        [x['query'] for x in q_items],
        bgem3_tokenizer, bgem3_model, device,
        max_length=CFG['eval_query_max_length'],
        batch_size=CFG['eval_query_batch_size'],
    )

    shot_to_idx = {s: i for i, s in enumerate(shot_ids)}
    filtered = [x for x in q_items if x['shot_id'] in shot_to_idx]
    q_idx = [q_items.index(x) for x in filtered]
    q_embs_f = q_embs_all[q_idx]
    gt_indices = [shot_to_idx[x['shot_id']] for x in filtered]
    tau = CFG['dual_softmax_tau']

    def _rank_metrics(d_embs, q_embs):
        sim_raw = torch.matmul(q_embs, d_embs.to(device).T) / tau
        sim_dsl = torch.softmax(sim_raw, dim=1) * torch.softmax(sim_raw, dim=0)
        sim_np = sim_dsl.detach().cpu().numpy()
        ranks = [
            int(np.where(np.argsort(-sim_np[i]) == gt)[0][0]) + 1
            for i, gt in enumerate(gt_indices)
        ]
        r = np.asarray(ranks)
        return {
            'R1':   100.0 * float(np.mean(r <= 1)),
            'R5':   100.0 * float(np.mean(r <= 5)),
            'R10':  100.0 * float(np.mean(r <= 10)),
            'MdR':  float(np.median(r)),
            'SumR': 100.0 * float(np.mean(r <= 1) + np.mean(r <= 5) + np.mean(r <= 10)),
        }

    # Full-dim metrics
    metrics = _rank_metrics(doc_embs, q_embs_f)

    # MRL per-dim metrics (truncate + re-normalize, reuse encoded queries)
    by_dim = {}
    for d in MRL_DIMS:
        d_trunc = F.normalize(doc_embs[:, :d].float(), p=2, dim=-1)
        q_trunc = F.normalize(q_embs_f[:, :d].float(), p=2, dim=-1)
        by_dim[d] = _rank_metrics(d_trunc, q_trunc)
    metrics['by_dim'] = by_dim

    return metrics


print('Training and evaluation utilities ready.')
print(f'MRL dims: {MRL_DIMS}')


Training and evaluation utilities ready.
MRL dims: (64, 256, 1024)


In [6]:
def run_ablation(
    model_class,
    loss_cfg,
    ckpt_name,
    desc,
    train_from_scratch=True,
    train_loader_override=None,
    val_loader_override=None,
    arch_override=None,
):
    """
    Train one ablation variant and return history + best val metrics.

    Args:
        model_class : FusionEncoder subclass to instantiate
        loss_cfg    : dict with lambda_narr, beta_hard, gamma_mrl
        ckpt_name   : filename (no path) for saving best checkpoint
        desc        : human-readable name for tqdm display
        train_from_scratch:
            True  -> train from epoch 1 (ignore old checkpoint/log)
            False -> if best checkpoint/log exists, load and continue training
        train_loader_override:
            Optional DataLoader for training (if None -> use global train_loader)
        val_loader_override:
            Optional DataLoader for validation (if None -> use global val_loader)
        arch_override:
            Optional dict to override architecture args (eg. vis_dim for CLIP study)
    """
    arch = dict(
        dim=CFG['dim'], vis_dim=CFG['vis_dim'],
        n_layers=CFG['n_layers'], n_heads=CFG['n_heads'], n_kv_heads=CFG['n_kv_heads'],
    )
    if arch_override is not None:
        arch.update(arch_override)

    active_train_loader = train_loader_override if train_loader_override is not None else train_loader

    model = model_class(**arch).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay']
    )
    total_steps = max(1, len(active_train_loader) * CFG['epochs'])
    warmup_steps = int(total_steps * CFG['warmup_ratio'])
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler = torch.amp.GradScaler('cuda' if DEVICE.type == 'cuda' else 'cpu')

    ckpt_path = ABLATION_OUTPUT / ckpt_name
    log_path = ABLATION_OUTPUT / ckpt_name.replace('.pth', '_log.json')

    best_r1 = -1.0
    history = []
    start_epoch = 1

    if not train_from_scratch:
        if ckpt_path.exists():
            state_dict = torch.load(ckpt_path, map_location=DEVICE)
            model.load_state_dict(state_dict, strict=True)
            print(f'[{desc}] Loaded existing best checkpoint: {ckpt_path}')
        else:
            print(f'[{desc}] No checkpoint found: {ckpt_path}. Train from scratch.')

        if log_path.exists():
            try:
                old_history = json.loads(log_path.read_text(encoding='utf-8'))
                if isinstance(old_history, list):
                    history = old_history
                    if history:
                        best_r1 = max(float(h.get('R1', -1.0)) for h in history)
                        start_epoch = min(len(history) + 1, CFG['epochs'] + 1)
                        print(
                            f'[{desc}] Resume from log: '
                            f'completed={len(history)} epochs, best R@1={best_r1:.2f}'
                        )
            except Exception as e:
                print(f'[{desc}] Cannot read resume log ({e}). Continue without log metadata.')

    if start_epoch > CFG['epochs']:
        print(
            f"[{desc}] Skip training: completed epochs ({len(history)}) "
            f">= CFG['epochs'] ({CFG['epochs']})."
        )
    else:
        for epoch in range(start_epoch, CFG['epochs'] + 1):
            model.train()
            running_loss = 0.0

            pbar = tqdm(
                active_train_loader, total=len(active_train_loader),
                desc=f'[{desc}] Epoch {epoch:02d}/{CFG["epochs"]}',
                leave=False, dynamic_ncols=True,
            )
            for step, batch in enumerate(pbar, start=1):
                q_hat = encode_queries(
                    batch['query_text'], bgem3_tokenizer, bgem3_model, DEVICE,
                    max_length=CFG['train_query_max_length'],
                )
                hard_neg_embs = _encode_hard_negs(
                    batch['hard_neg_texts'], bgem3_tokenizer, bgem3_model, DEVICE,
                    CFG['train_query_max_length'],
                )

                amp_ctx = torch.amp.autocast(
                    device_type='cuda' if DEVICE.type == 'cuda' else 'cpu',
                    enabled=(DEVICE.type == 'cuda'), dtype=torch.bfloat16,
                )
                with amp_ctx:
                    e_plus, e_narr, temperature = model(
                        seg_tokens=batch['token_reprs'].to(DEVICE),
                        seg_pooled=batch['segment_pooled'].to(DEVICE),
                        visual_features=batch['visual_features'].to(DEVICE),
                        seg_timestamps=batch['seg_timestamps'].to(DEVICE),
                        frame_timestamps=batch['frame_timestamps'].to(DEVICE),
                        seg_mask=batch['segment_mask'].to(DEVICE),
                        visual_mask=batch['visual_mask'].to(DEVICE),
                        query_emb=q_hat,
                        token_mask=batch['token_mask'].to(DEVICE),
                    )
                    loss = retrieval_loss(
                        q_hat, e_plus, e_narr, temperature,
                        hard_neg_embs=hard_neg_embs,
                        **loss_cfg,
                    )

                optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()

                running_loss += float(loss.item())
                pbar.set_postfix(loss=f'{running_loss / step:.4f}')

            train_loss = running_loss / len(active_train_loader)
            if DEVICE.type == 'cuda':
                gc.collect()
                torch.cuda.empty_cache()

            if val_loader_override is None:
                val_metrics = evaluate_on_val(model, DEVICE)
            else:
                old_val_loader = globals().get('val_loader', None)
                try:
                    globals()['val_loader'] = val_loader_override
                    val_metrics = evaluate_on_val(model, DEVICE)
                finally:
                    if old_val_loader is None:
                        globals().pop('val_loader', None)
                    else:
                        globals()['val_loader'] = old_val_loader

            history.append({'epoch': epoch, 'train_loss': train_loss, **val_metrics})

            if val_metrics['R1'] > best_r1:
                best_r1 = val_metrics['R1']
                torch.save(model.state_dict(), ckpt_path)

            print(
                f"[{desc}] Epoch {epoch:02d}/{CFG['epochs']} "
                f"| loss={train_loss:.4f} "
                f"| val R@1={val_metrics['R1']:.2f} "
                f"| R@5={val_metrics['R5']:.2f} "
                f"| R@10={val_metrics['R10']:.2f}"
            )

    log_path.write_text(json.dumps(history, indent=2), encoding='utf-8')

    del model
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    print(f'[{desc}] Best Val R@1: {best_r1:.2f}  | Saved: {ckpt_path}')
    return history, best_r1


print('run_ablation() ready.')

run_ablation() ready.


## Ablation 1: Tháo Adaptive Filtering

Thay adaptive filtering + weighted pooling bằng **uniform pooling** trên tất cả valid segments.
Loss giữ nguyên (lambda_narr=0.5, gamma_mrl=0.1).

In [7]:
history_a1, best_r1_a1 = run_ablation(
    model_class=FusionEncoderNoFilter,
    loss_cfg=dict(lambda_narr=0.5, beta_hard=0.3, gamma_mrl=0.1),
    ckpt_name='ablation1_no_filter.pth',
    desc='Ablation1-NoFilter',
)

[Ablation1-NoFilter] Epoch 01/20 | loss=2.2181 | val R@1=30.43 | R@5=52.45 | R@10=61.58


[Ablation1-NoFilter] Epoch 02/20 | loss=2.2096 | val R@1=37.54 | R@5=60.45 | R@10=68.55


[Ablation1-NoFilter] Epoch 03/20 | loss=2.2033 | val R@1=45.07 | R@5=68.63 | R@10=76.18


[Ablation1-NoFilter] Epoch 04/20 | loss=2.1949 | val R@1=50.27 | R@5=72.75 | R@10=79.87


[Ablation1-NoFilter] Epoch 05/20 | loss=2.1839 | val R@1=54.96 | R@5=76.86 | R@10=82.90


...

[Ablation1-NoFilter] Epoch 16/20 | loss=1.9525 | val R@1=71.96 | R@5=88.59 | R@10=92.15


[Ablation1-NoFilter] Epoch 17/20 | loss=1.9434 | val R@1=72.18 | R@5=88.63 | R@10=92.14


[Ablation1-NoFilter] Epoch 18/20 | loss=1.9374 | val R@1=72.29 | R@5=88.69 | R@10=92.25


[Ablation1-NoFilter] Epoch 19/20 | loss=1.9351 | val R@1=72.25 | R@5=88.71 | R@10=92.28


[Ablation1-NoFilter] Epoch 20/20 | loss=1.9341 | val R@1=72.25 | R@5=88.69 | R@10=92.28
[Ablation1-NoFilter] Best Val R@1: 72.29  | Saved: /media/urlab/KINGSTON/aic/source/ablation_output/ablation1_no_filter.pth


## Ablation 3: Tháo Narration Auxiliary Loss

Đặt `lambda_narr = 0.0` → không tính `L_narr = InfoNCE(q_hat, e_narr)`.
Model và phần còn lại của loss giữ nguyên.

In [8]:
history_a3, best_r1_a3 = run_ablation(
    model_class=FusionEncoder,
    loss_cfg=dict(lambda_narr=0.0, beta_hard=0.3, gamma_mrl=0.1),
    ckpt_name='ablation3_no_narr.pth',
    desc='Ablation3-NoNarrLoss',
)

[Ablation3-NoNarrLoss] Epoch 01/20 | loss=1.5281 | val R@1=30.06 | R@5=51.51 | R@10=60.75


[Ablation3-NoNarrLoss] Epoch 02/20 | loss=1.5198 | val R@1=36.75 | R@5=58.86 | R@10=67.33


[Ablation3-NoNarrLoss] Epoch 03/20 | loss=1.5137 | val R@1=42.91 | R@5=65.70 | R@10=74.02


[Ablation3-NoNarrLoss] Epoch 04/20 | loss=1.5053 | val R@1=47.48 | R@5=69.02 | R@10=76.22


[Ablation3-NoNarrLoss] Epoch 05/20 | loss=1.4947 | val R@1=39.43 | R@5=59.73 | R@10=67.22


...

[Ablation3-NoNarrLoss] Epoch 16/20 | loss=1.2703 | val R@1=58.71 | R@5=79.61 | R@10=84.84


[Ablation3-NoNarrLoss] Epoch 17/20 | loss=1.2615 | val R@1=58.71 | R@5=79.45 | R@10=84.59


[Ablation3-NoNarrLoss] Epoch 18/20 | loss=1.2556 | val R@1=58.53 | R@5=79.64 | R@10=84.41


[Ablation3-NoNarrLoss] Epoch 19/20 | loss=1.2518 | val R@1=58.52 | R@5=79.54 | R@10=84.55


[Ablation3-NoNarrLoss] Epoch 20/20 | loss=1.2512 | val R@1=58.54 | R@5=79.53 | R@10=84.57
[Ablation3-NoNarrLoss] Best Val R@1: 58.71  | Saved: /media/urlab/KINGSTON/aic/source/ablation_output/ablation3_no_narr.pth


## Ablation 4: Tháo Gated Attention

Thay `GatedAttention` (sigmoid gate init -5.0) bằng standard multi-head attention.
Loss giữ nguyên.

In [9]:
history_a4, best_r1_a4 = run_ablation(
    model_class=FusionEncoderNoGate,
    loss_cfg=dict(lambda_narr=0.5, beta_hard=0.3, gamma_mrl=0.1),
    ckpt_name='ablation4_no_gate.pth',
    desc='Ablation4-NoGate',
)

[Ablation4-NoGate] Epoch 01/20 | loss=2.2180 | val R@1=32.04 | R@5=55.00 | R@10=64.15


[Ablation4-NoGate] Epoch 02/20 | loss=2.2085 | val R@1=46.01 | R@5=69.18 | R@10=76.90


[Ablation4-NoGate] Epoch 03/20 | loss=2.2019 | val R@1=52.68 | R@5=75.31 | R@10=82.26


[Ablation4-NoGate] Epoch 04/20 | loss=2.1929 | val R@1=56.13 | R@5=77.68 | R@10=84.05


[Ablation4-NoGate] Epoch 05/20 | loss=2.1812 | val R@1=59.77 | R@5=80.37 | R@10=86.22


...

[Ablation4-NoGate] Epoch 16/20 | loss=1.9372 | val R@1=66.17 | R@5=83.63 | R@10=87.84


[Ablation4-NoGate] Epoch 17/20 | loss=1.9267 | val R@1=66.20 | R@5=83.50 | R@10=87.73


[Ablation4-NoGate] Epoch 18/20 | loss=1.9193 | val R@1=66.17 | R@5=83.58 | R@10=87.78


[Ablation4-NoGate] Epoch 19/20 | loss=1.9160 | val R@1=66.08 | R@5=83.31 | R@10=87.63


[Ablation4-NoGate] Epoch 20/20 | loss=1.9151 | val R@1=65.99 | R@5=83.36 | R@10=87.62
[Ablation4-NoGate] Best Val R@1: 67.05  | Saved: /media/urlab/KINGSTON/aic/source/ablation_output/ablation4_no_gate.pth


## MRL Test: Tháo MRL Loss

Đặt `gamma_mrl = 0.0` → không tính loss trên các chiều truncated [64, 128, 256, 512, 1024].
Kiểm chứng MRL có cải thiện representation quality không.

In [10]:
history_no_mrl, best_r1_no_mrl = run_ablation(
    model_class=FusionEncoder,
    loss_cfg=dict(lambda_narr=0.5, beta_hard=0.3, gamma_mrl=0.0),
    ckpt_name='ablation_no_mrl.pth',
    desc='NoMRL',
)

[NoMRL] Epoch 01/20 | loss=2.0796 | val R@1=30.26 | R@5=52.47 | R@10=61.81


[NoMRL] Epoch 02/20 | loss=2.0712 | val R@1=37.92 | R@5=60.53 | R@10=69.24


[NoMRL] Epoch 03/20 | loss=2.0641 | val R@1=44.16 | R@5=66.30 | R@10=74.38


[NoMRL] Epoch 04/20 | loss=2.0550 | val R@1=47.80 | R@5=70.48 | R@10=77.34


[NoMRL] Epoch 05/20 | loss=2.0431 | val R@1=45.20 | R@5=66.07 | R@10=72.92


...

[NoMRL] Epoch 16/20 | loss=1.8029 | val R@1=59.73 | R@5=81.21 | R@10=86.21


[NoMRL] Epoch 17/20 | loss=1.7910 | val R@1=59.82 | R@5=81.37 | R@10=86.14


[NoMRL] Epoch 18/20 | loss=1.7871 | val R@1=59.79 | R@5=81.00 | R@10=85.88


[NoMRL] Epoch 19/20 | loss=1.7847 | val R@1=59.82 | R@5=81.25 | R@10=86.01


[NoMRL] Epoch 20/20 | loss=1.7830 | val R@1=59.90 | R@5=81.23 | R@10=86.04
[NoMRL] Best Val R@1: 59.90  | Saved: /media/urlab/KINGSTON/aic/source/ablation_output/ablation_no_mrl.pth


## Study Model: CLIP-B/16 Layer -2 + No Gate

Case này chỉ chạy **1 study model**:
- Visual features dùng CLIP ViT-B/16 (layer -2)
- Tháo Gated Attention (`NoGate`)
- Train lại từ đầu theo `CFG['clip_train_from_scratch']`

In [8]:
if clip_train_loader is None or clip_val_loader is None:
    raise RuntimeError(
        f'CLIP loaders are not ready. Missing features in: {CLIP_FEATURE_DIR}. '
        'Run source/precompute_clip_patch16.py first.'
    )

CLIP_TRAIN_FROM_SCRATCH = bool(CFG.get('clip_train_from_scratch', True))

history_clip_nogate, best_r1_clip_nogate = run_ablation(
    model_class=FusionEncoderNoGate,
    loss_cfg=dict(lambda_narr=0.5, beta_hard=0.3, gamma_mrl=0.1),
    ckpt_name='study_clip_b16_layer2_no_gate.pth',
    desc='Study-CLIPB16L2-NoGate',
    train_from_scratch=CLIP_TRAIN_FROM_SCRATCH,
    train_loader_override=clip_train_loader,
    val_loader_override=clip_val_loader,
    arch_override=dict(vis_dim=CFG['clip_vis_dim']),
)

clip_study_ckpt = ABLATION_OUTPUT / 'study_clip_b16_layer2_no_gate.pth'
clip_arch = dict(
    dim=CFG['dim'],
    vis_dim=CFG['clip_vis_dim'],
    n_layers=CFG['n_layers'],
    n_heads=CFG['n_heads'],
    n_kv_heads=CFG['n_kv_heads'],
)

clip_nogate_metrics = None
if clip_study_ckpt.exists():
    m = FusionEncoderNoGate(**clip_arch).to(DEVICE)
    m.load_state_dict(torch.load(clip_study_ckpt, map_location=DEVICE))
    m.eval()

    _old_val_loader = val_loader
    try:
        val_loader = clip_val_loader
        clip_nogate_metrics = evaluate_on_val(m, DEVICE)
    finally:
        val_loader = _old_val_loader

    del m
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

    print('\n[Study-CLIPB16L2-NoGate] Re-eval on CLIP val loader:')
    print(
        f"  R@1={clip_nogate_metrics['R1']:.2f}  "
        f"R@5={clip_nogate_metrics['R5']:.2f}  "
        f"R@10={clip_nogate_metrics['R10']:.2f}  "
        f"SumR={clip_nogate_metrics['SumR']:.2f}"
    )

    clip_eval_json = ABLATION_OUTPUT / 'study_clip_b16_layer2_no_gate_eval.json'
    clip_eval_json.write_text(
        json.dumps(clip_nogate_metrics, indent=2, ensure_ascii=False),
        encoding='utf-8',
    )
    print(f'  Saved eval -> {clip_eval_json}')
else:
    print(f'[WARN] Study checkpoint not found: {clip_study_ckpt}')

[Study-CLIPB16L2-NoGate] Epoch 01/20 | loss=2.2179 | val R@1=32.81 | R@5=54.63 | R@10=63.79


[Study-CLIPB16L2-NoGate] Epoch 02/20 | loss=2.2084 | val R@1=46.99 | R@5=70.26 | R@10=77.13


[Study-CLIPB16L2-NoGate] Epoch 03/20 | loss=2.2017 | val R@1=53.19 | R@5=75.28 | R@10=81.64


[Study-CLIPB16L2-NoGate] Epoch 04/20 | loss=2.1923 | val R@1=57.23 | R@5=78.46 | R@10=84.70


[Study-CLIPB16L2-NoGate] Epoch 05/20 | loss=2.1804 | val R@1=59.02 | R@5=79.49 | R@10=84.72


...

[Study-CLIPB16L2-NoGate] Epoch 17/20 | loss=1.9238 | val R@1=63.35 | R@5=80.97 | R@10=86.01


[Study-CLIPB16L2-NoGate] Epoch 18/20 | loss=1.9165 | val R@1=62.89 | R@5=80.57 | R@10=85.53


[Study-CLIPB16L2-NoGate] Epoch 19/20 | loss=1.9136 | val R@1=63.21 | R@5=81.07 | R@10=85.85


[Study-CLIPB16L2-NoGate] Epoch 20/20 | loss=1.9136 | val R@1=63.06 | R@5=80.90 | R@10=85.76
[Study-CLIPB16L2-NoGate] Best Val R@1: 64.87  | Saved: /media/urlab/KINGSTON/aic/source/ablation_output/study_clip_b16_layer2_no_gate.pth



[Study-CLIPB16L2-NoGate] Re-eval on CLIP val loader:
  R@1=64.87  R@5=82.85  R@10=87.24  SumR=234.95
  Saved eval -> /media/urlab/KINGSTON/aic/source/ablation_output/study_clip_b16_layer2_no_gate_eval.json


## So sánh tất cả variants trên Val split

In [ ]:

arch = dict(
    dim=CFG['dim'], vis_dim=CFG['vis_dim'],
    n_layers=CFG['n_layers'], n_heads=CFG['n_heads'], n_kv_heads=CFG['n_kv_heads'],
)

VARIANTS = [
    ('Full Model (baseline)',                    FusionEncoder,               FULL_MODEL_CKPT),
    ('Ablation 1 - No AdaptFilter',              FusionEncoderNoFilter,       ABLATION_OUTPUT / 'ablation1_no_filter.pth'),
    ('Ablation 3 - No NarrLoss',                 FusionEncoder,               ABLATION_OUTPUT / 'ablation3_no_narr.pth'),
    ('Ablation 4 - No Gate',                     FusionEncoderNoGate,         ABLATION_OUTPUT / 'ablation4_no_gate.pth'),
    ('Combo - No AdaptFilter + No NarrLoss',     FusionEncoderNoFilter,       ABLATION_OUTPUT / 'ablation_combo_no_filter_no_narr.pth'),
    ('Combo - No AdaptFilter + No Gate',         FusionEncoderNoFilterNoGate, ABLATION_OUTPUT / 'ablation_combo_no_filter_no_gate.pth'),
    ('MRL Test - No MRL Loss',                   FusionEncoder,               ABLATION_OUTPUT / 'ablation_no_mrl.pth'),
]

eval_results = {}
for name, model_class, ckpt_path in VARIANTS:
    ckpt_path = Path(ckpt_path)
    if not ckpt_path.exists():
        print(f'  SKIP {name} - checkpoint not found: {ckpt_path}')
        continue
    print(f'Evaluating: {name}')
    m = model_class(**arch).to(DEVICE)
    m.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    m.eval()
    metrics = evaluate_on_val(m, DEVICE)
    eval_results[name] = metrics
    del m
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    mrl_str = '  '.join(
        f"d{d}={metrics['by_dim'][d]['R1']:.2f}" for d in MRL_DIMS
    )
    print(
        f"  Full: R@1={metrics['R1']:.2f}  R@5={metrics['R5']:.2f}  "
        f"R@10={metrics['R10']:.2f}  SumR={metrics['SumR']:.2f}"
    )
    print(f"  MRL R@1 -> {mrl_str}")

out_json = ABLATION_OUTPUT / 'ablation_eval_results.json'
out_json.write_text(json.dumps(eval_results, indent=2, ensure_ascii=False), encoding='utf-8')
print(f'\nSaved eval results -> {out_json}')


Evaluating: Full Model (baseline)


  Full: R@1=59.27  R@5=80.16  R@10=85.13  SumR=224.56
  MRL R@1 → d64=34.47  d256=58.90  d1024=59.27
Evaluating: Ablation 1 – No AdaptFilter


  Full: R@1=72.52  R@5=88.76  R@10=92.34  SumR=253.62
  MRL R@1 → d64=45.49  d256=69.37  d1024=72.52
Evaluating: Ablation 3 – No NarrLoss


  Full: R@1=58.71  R@5=79.61  R@10=84.84  SumR=223.16
  MRL R@1 → d64=34.15  d256=58.32  d1024=58.71
Evaluating: Ablation 4 – No Gate


  Full: R@1=67.05  R@5=84.30  R@10=88.28  SumR=239.62
  MRL R@1 → d64=41.92  d256=64.48  d1024=67.05
Evaluating: MRL Test   – No MRL Loss


  Full: R@1=59.90  R@5=81.23  R@10=86.04  SumR=227.17
  MRL R@1 → d64=35.35  d256=59.63  d1024=59.90

Saved eval results → /media/urlab/KINGSTON/aic/source/ablation_output/ablation_eval_results.json


In [ ]:

if not eval_results:
    print('No results to display. Run evaluation cell first.')
else:
    full = eval_results.get('Full Model (baseline)', {})

    # Header: R@1 (full) + MRL R@1 per dim + dR@1
    dim_headers = ''.join(f"{'d'+str(d):>7}" for d in MRL_DIMS)
    header = f"{'Variant':<50} {'R@1':>6} {'SumR':>8}{dim_headers}  {'dR@1':>7}"
    sep = '-' * len(header)
    print(sep)
    print(header)
    print(sep)

    for name, m in eval_results.items():
        dr1 = m['R1'] - full.get('R1', 0) if name != 'Full Model (baseline)' else 0.0
        sign = f'{dr1:+.2f}' if name != 'Full Model (baseline)' else '  base'
        mrl_cols = ''
        for d in MRL_DIMS:
            r1_d = m.get('by_dim', {}).get(d, {}).get('R1', float('nan'))
            mrl_cols += f'{r1_d:>7.2f}'
        print(
            f"{name:<50} {m['R1']:>6.2f} {m['SumR']:>8.2f}{mrl_cols}  {sign:>7}"
        )

    print(sep)
    print()
    print(f"Columns: R@1=full {CFG['dim']}d  |  MRL R@1 at dims {MRL_DIMS}  |  dR@1 vs Full Model")
    print()
    print('Giải thích dR@1:')
    print('  Âm  -> tháo thành phần này làm giảm R@1 -> thành phần CÓ ích')
    print('  Dương -> tháo thành phần này làm TĂNG R@1 -> thành phần KHÔNG cần thiết')
    print()
    print('Giải thích MRL dims:')
    print('  d64  = embedding truncated xuống 64 chiều  (nhỏ nhất)')
    print('  d256 = embedding truncated xuống 256 chiều (giữa)')
    print('  d1024= embedding đầy đủ                    (lớn nhất)')
    print()
    print('MRL Test - No MRL Loss vs Full Model:')
    no_mrl = eval_results.get('MRL Test - No MRL Loss', {})
    if no_mrl:
        for d in MRL_DIMS:
            r1_full = full.get('by_dim', {}).get(d, {}).get('R1', float('nan'))
            r1_nomrl = no_mrl.get('by_dim', {}).get(d, {}).get('R1', float('nan'))
            diff = r1_full - r1_nomrl
            verdict = 'MRL giúp ích' if diff > 0.5 else ('không đáng kể' if abs(diff) <= 0.5 else 'MRL không giúp')
            print(f'  dim {d:>4}: Full={r1_full:.2f}  NoMRL={r1_nomrl:.2f}  diff={diff:+.2f}  -> {verdict}')


---------------------------------------------------------------------------------
Variant                                R@1     SumR    d64   d256  d1024     dR@1
---------------------------------------------------------------------------------
Full Model (baseline)                59.27   224.56  34.47  58.90  59.27     base
Ablation 1 – No AdaptFilter          72.52   253.62  45.49  69.37  72.52   +13.25
Ablation 3 – No NarrLoss             58.71   223.16  34.15  58.32  58.71    -0.56
Ablation 4 – No Gate                 67.05   239.62  41.92  64.48  67.05    +7.77
MRL Test   – No MRL Loss             59.90   227.17  35.35  59.63  59.90    +0.63
---------------------------------------------------------------------------------

Columns: R@1=full 1024d  |  MRL R@1 at dims (64, 256, 1024)  |  dR@1 vs Full Model

Giải thích dR@1:
  Âm  → tháo thành phần này làm giảm R@1 → thành phần CÓ ích
  Dương → tháo thành phần này làm TĂNG R@1 → thành phần KHÔNG cần thiết

Giải thích MRL dims:
  d64